In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

import sys
sys.path.insert(0, "../../packages/ProMERGE_utils")
from promerge_utils import concat_emb

In [ ]:
os.makedirs("../../result/U2OS_HEK293T/input/", exist_ok=True)

In [ ]:
U2OS_img = pd.read_csv("../../data/U2OS/image_emd_1.tsv", sep="\t")
U2OS_apms = pd.read_csv("../../data/U2OS/ppi_emd.tsv", sep="\t")
HEK293_img = pd.read_csv("../../data/OpenCell/image_emd_1024pcs.tsv", sep="\t", index_col=0)
HEK293_apms = pd.read_csv("../../data/OpenCell/ppi_emd.tsv", sep="\t")

In [ ]:
# fix APMS genes
mapping = {
    "2021-09-15 00:00:00": "SELENOF",
    "2021-03-06 00:00:00": "MARCHF6",
    "2021-03-07 00:00:00": "MARCHF7",
    "2021-03-08 00:00:00": "MARCHF8"
}
HEK293_apms["index"] = HEK293_apms.index.values
HEK293_apms.index = HEK293_apms["index"].replace(mapping)
del HEK293_apms["index"]

In [ ]:
U2OS_img.columns = range(1, 1025)
U2OS_img.index.name = 0
U2OS_img.reset_index().to_csv("../../result/U2OS_HEK293T/input/Image-U2OS.tsv", sep="\t", index=False)

U2OS_apms.columns = range(1, 1025)
U2OS_apms.index.name = 0
U2OS_apms.reset_index().to_csv("../../result/U2OS_HEK293T/input/APMS-U2OS.tsv", sep="\t", index=False)

HEK293_img.columns = range(1, 1025)
HEK293_img.index.name = 0
HEK293_img.reset_index().to_csv("../../result/U2OS_HEK293T/input/Image-HEK293T.tsv", sep="\t", index=False)

HEK293_apms.columns = range(1, 1025)
HEK293_apms.index.name = 0
HEK293_apms.reset_index().to_csv("../../result/U2OS_HEK293T/input/APMS-HEK293T.tsv", sep="\t", index=False)

### Stats for future filtering

In [ ]:
# load genes in each experiment
ap_u2os = set(pd.read_csv("../../result/U2OS_HEK293T/input/APMS-U2OS.tsv", sep="\t", index_col=0).index)
ap_hek = set(pd.read_csv("../../result/U2OS_HEK293T/input/APMS-HEK293T.tsv", sep="\t", index_col=0).index)
img_u2os = set(pd.read_csv("../../result/U2OS_HEK293T/input/Image-U2OS.tsv", sep="\t", index_col=0).index)
img_hek = set(pd.read_csv("../../result/U2OS_HEK293T/input/Image-HEK293T.tsv", sep="\t", index_col=0).index)

In [ ]:
# total possible pairs in each cell line
u2os_genes = ap_u2os | img_u2os
u2os_pairs = len(u2os_genes) * (len(u2os_genes)-1) // 2

hek_genes = ap_hek | img_hek
hek_pairs = len(hek_genes) * (len(hek_genes)-1) // 2

In [ ]:
# top 1%, add other when needed
stat = pd.DataFrame(
    [
        [int(u2os_pairs*0.01), int(hek_pairs*0.01)]
    ],
    index=["Percent1"],
    columns=["U2OS", "HEK293T"]
)
stat.to_csv("../../result/U2OS_HEK293T/input/pair_stat.tsv", sep="\t")

### Concatenate the modality embeddings

In [ ]:
df1 = pd.read_csv(
    "../../result/U2OS_HEK293T/input/APMS-U2OS.tsv",
    sep="\t",
    index_col=0
)
df2 = pd.read_csv(
    "../../result/U2OS_HEK293T/input/Image-U2OS.tsv",
    sep="\t",
    index_col=0
)
df_concat = concat_emb([df1, df2])
df_concat.to_csv(
    "../../result/U2OS_HEK293T/input/Concat-U2OS.tsv",
    sep="\t",
    header=True,
    index=True
)

In [ ]:
df1 = pd.read_csv(
    "../../result/U2OS_HEK293T/input/APMS-HEK293T.tsv",
    sep="\t",
    index_col=0
)
df2 = pd.read_csv(
    "../../result/U2OS_HEK293T/input/Image-HEK293T.tsv",
    sep="\t",
    index_col=0
)
df_concat = concat_emb([df1, df2])
df_concat.to_csv(
    "../../result/U2OS_HEK293T/input/Concat-HEK293T.tsv",
    sep="\t",
    header=True,
    index=True
)